In [1]:
import os
import json
import random
from typing import Any, Dict, List, Union
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv
from underthesea import word_tokenize

import asyncio
from typing import List, Dict, Any, Awaitable

from pydantic import BaseModel
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, Retrying, retry_if_exception_type, before_sleep_log, after_log
import logging
import aiofiles

# Configure logging for the retry mechanism
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

/home/octoopt/workspace/projects/personal/data_enrichment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY, 
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/" 
)


In [3]:
# Define the type alias for clarity
JsonData = Union[Dict[str, Any], List[Any]]

def write_json(filepath: str, data: JsonData) -> bool:
    """
    Writes data to a specified JSON file path.

    Args:
        filepath: The path to the output JSON file.
        data: The Python data structure (dict or list) to save.

    Returns:
        True if the write was successful, False otherwise.
    """
    try:
        # Open file in write mode ('w'), specifying UTF-8 encoding
        with open(filepath, 'w', encoding='utf-8') as f:
            # Use json.dump for writing. indent=4 makes the file human-readable.
            json.dump(data, f, indent=4)
        print(f"Successfully wrote data to {filepath}")
        return True
    except IOError as e:
        print(f"Error writing to file {filepath}: {e}")
        return False
    except TypeError as e:
        print(f"Data type error during serialization: {e}. Check if data is JSON serializable.")
        return False

def read_json(filepath: str) -> Union[JsonData, None]:
    """
    Reads and parses data from a specified JSON file path.

    Args:
        filepath: The path to the input JSON file.

    Returns:
        The Python data structure (dict or list) loaded from the file,
        or None if an error occurred.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None

    try:
        # Open file in read mode ('r'), specifying UTF-8 encoding
        with open(filepath, 'r', encoding='utf-8') as f:
            # Use json.load to parse the JSON content
            data = json.load(f)
            print(f"Successfully read data from {filepath}")
            return data
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from {filepath}. File might be empty or corrupted: {e}")
        return None
    except IOError as e:
        print(f"Error reading file {filepath}: {e}")
        return None

In [4]:
DATASET_IDS = [
    "OpenHust/vietnamese-summarization",
    "HaiLong9901/VietNameseLongTextSum",
    "truongpdd/vietnamese_story"
]

In [9]:
ds_1 = load_dataset(DATASET_IDS[0])
ds_2 = load_dataset(DATASET_IDS[1])

Generating train split: 74564 examples [00:04, 17468.58 examples/s]
Generating test split: 100%|██████████| 500/500 [00:00<00:00, 3358.03 examples/s]


In [10]:
ds_1

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [13]:
ds_1['train'][0]

{'Unnamed: 0': 0,
 'Document': 'Đây là một trong những nội dung tại văn bản vừa được UBND TP Hà Nội ban hành về việc tăng cường công tác quản lý nuôi , giết mổ , kinh doanh và sử dụng thịt chó , mèo trên địa bàn .Theo đó , các sở , ngành trên địa bàn phải vào cuộc ngay để hướng tới thay đổi thói quen của người dân khi dùng chó , mèo làm thực phẩm .Gây phản cảm với du khách , người nước ngoàiCũng trong văn bản này , UBND TP Hà Nội thừa nhận rằng việc kinh doanh , giết mổ và sử dụng thịt chó , mèo tại Hà Nội thời gian qua đã tạo ra những hình ảnh phản cảm đối với khách tham quan du lịch , người nước ngoài đang sinh sống và làm việc tại Hà Nội , gây ảnh hưởng tới hình ảnh của một thủ đô " văn minh , hiện đại " .Trong thực tế , theo ghi nhận của Tuổi Trẻ tại phố Tam Trinh , ngay đoạn đầu cầu Mai Động ( quận Hoàng Mai ) , một đoạn phố dài với gần chục cửa hàng buôn bán thịt chó sống , nhà hàng phục vụ các món chế biến từ thịt chó vẫn hoạt động tấp nập nhiều năm nay .Chị Th . , một chủ sạp b

In [24]:
doc_d1 = []
for data in ds_1['train']:
    doc_d1.append(data['Document'])
len(doc_d1)

74564

In [20]:
ds_2['train'][15]

{'abstract': 'Tuần báo Bấm The Economist của Anh vừa đặt câu hỏi liệu tân Trưởng Ban nội chính có phải là cứu tinh mà Việt Nam chờ đợi.',
 'section_names': "Ông Bá Thanh sẽ chỉ 'gọt giũa' chút ít?",
 'article': 'Bài viết đăng ngày 25/1 với tựa đề "Ông Thanh có phải là cứu tinh", mở đầu với lời nhận xét về tình hình hiện tại của Việt Nam - đất nước điều hành bởi "một đảng đầy những vụ tai tiếng đang tìm cách loại bỏ giới bất đồng chính kiến và giải quyết nạn tham nhũng." Mới nhất trong số những vụ đàn áp giới bất đồng chính kiến này được dẫn ví dụ, đó là việc "tòa án Việt Nam tuyên án các bản tù dài hạn lên 14 nhà hoạt động dân chủ và blooger, dựa trên những bằng chứng mơ hồ về tội lật đổ chính quyền." \'Khủng long trái chiều\' Việc sử dụng phiên tòa nhằm thể hiện sức mạnh chính trị và đàn áp bất kỳ sự chống đối nào, theo The Economist, cho thấy "một hành động tuyệt vọng của Đảng, bởi chứng bệnh hoang tưởng ngày càng nặng." "Bất chấp những phát triển về kinh tế, thông qua một

In [25]:
doc_d2 = []
for key in ds_2.keys():
    dataset = ds_2[key]
    for data in dataset:
        _data = f"{data['abstract']} {data['article']}"
        doc_d2.append(_data)

len(doc_d2)


3000

In [26]:
import random

merged_data = doc_d1 + doc_d2 

random.shuffle(merged_data)


print(merged_data[0])
print(len(merged_data))

Dùng gạc vô trùng ấn lên lợi sau khi nhổ răng để giảm đau và cầm máu. Nếu lợi bị đau hoặc chảy máu một chút sau khi chiếc răng đã được nhổ, bạn hãy cuộn một miếng gạc vô trùng và ấn lên hốc răng (vùng lợi nơi chiếc răng đã nhổ). Ấn lên lợi cho đến khi hết chảy máu. Máu sẽ ngừng chảy trong khoảng vài phút. Bạn cũng có thể dùng túi trà ướt để xoa dịu lợi sau khi nhổ răng. Ngâm một túi trà vào nước nóng trong vài phút, sau đó lấy ra và vắt bớt nước. Chờ vài phút cho túi trà nguội và đặt lên hốc răng vừa nhổ để giảm cảm giác đau. Bạn có thể dùng trà xanh, trà đen, trà bạc hà cay hoặc trà hoa cúc chamomile để làm dịu đau. Nếu vẫn bị cơn đau làm phiền, bạn có thể uống thuốc giảm đau như acetaminophen hoặc ibuprofen. Nhớ đọc kỹ hướng dẫn sử dụng trên nhãn hộp thuốc. Nếu chiếc răng lung lay làm bạn bị đau hoặc có vẻ không thể nhổ được tại nhà, bạn hãy gọi cho nha sĩ để hẹn ngày đến phòng khám. Nha sĩ có thể nhổ chiếc răng với sự trợ giúp của thuốc tê để bạn không cảm thấy đau chút nào. Trong m

In [30]:
save_json = {
    "document": merged_data
}

save_data_path = "../data/vn_plain_dataset.json"
write_json(save_data_path, save_json)

Successfully wrote data to ../data/vn_plain_dataset.json


True

In [5]:
merged_data = read_json(filepath="../data/vn_plain_dataset.json")['document']

Successfully read data from ../data/vn_plain_dataset.json


## Build Summarization

In [6]:
from pydantic import BaseModel

class SummarizationSchema(BaseModel):
    summarized_document: str
    keywords: List[str]

SYSTEM_PROMPT = """
Bạn là chuyên gia tóm tắt tài liệu.

## ĐẦU RA YÊU CẦU

**TÓM TẮT** (5-7 câu):
1. Giới thiệu chủ đề chính
2. Nội dung/luận điểm quan trọng
3. Kết luận

**TỪ KHÓA** (5-7 từ):
Khái niệm chính, tên riêng, thuật ngữ chuyên ngành

## QUY TẮC
- Chính xác, súc tích
- Từ khóa phải có trong tài liệu gốc
- Định dạng: phân cách bằng dấu phẩy
- Đầu ra là tiếng Việt
"""

@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
async def summarize_document(llm: AsyncOpenAI, 
                        document: str,
                        model_name: str = 'gemini-2.0-flash-lite', 
                        system_prompt: str = SYSTEM_PROMPT) -> Dict:
    res = await llm.beta.chat.completions.parse(
    model=model_name,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": document},
    ],
    response_format=SummarizationSchema,
)
    parsed_res = res.choices[0].message.parsed
    return {
        "document": document,
        "summary": parsed_res.summarized_document,
        "keywords": parsed_res.keywords,
    }

In [7]:
await summarize_document(llm=client, document=merged_data[10])

2025-10-25 12:49:20,504 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


{'document': 'Theo V Cục N Thú y N , đến nay tổng số N lợn N tiêu hủy A là hơn A 23 . 000 con N và dịch N chủ yếu A xuất hiện V tại các hộ N chăn nuôi V nhỏ lẻ A , điều kiện N vệ sinh V và an toàn A sinh học N không tốt A , chưa xuất hiện V tại các trang trại N quy mô N lớn A . Cục N nhận định V , nguy cơ N dịch V tiếp tục V lan V rộng A là rất cao A . Lực lượng N chức năng N Hà Nội diễn tập V ứng phó N dịch tả V lợn N Châu Phi ngày N 7/3 . Về N nguyên nhân N dịch V lây lan N , Cục trưởng N Thú y N Phạm Văn Đông nói V kết quả N điều tra V bước đầu N xác định V nguyên nhân N chính là V một số người N chăn nuôi V , thương lái N chưa nhận thức V đầy đủ A tính chất N nguy hiểm A của dịch bệnh N , vì lợi ích N kinh tế N trước mắt N nên đã mua bán V , vận chuyển V , giết mổ V , tiêu thụ V lợn N bệnh N , lợn N nghi V mắc V bệnh N . Trong khi N đó , các cơ sở N nhỏ lẻ A có V mật độ N chăn nuôi V cao A , hộ N chăn nuôi V lợn N đan xen V trong khu N dân cư N không thường xuyên A thực hiện V biện

In [13]:
import asyncio
import aiofiles
import json
import random
import logging
from pathlib import Path
from typing import List, Dict, Any
from tqdm.asyncio import tqdm_asyncio

# Silence noisy logs
for name in ["httpx", "httpcore", "openai", "asyncopenai", "urllib3"]:
    logging.getLogger(name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


async def append_jsonl(result: Dict[str, Any], filename: Path):
    """
    Appends a single JSON record as one line (JSONL format).
    This is safe for concurrent writes.
    """
    try:
        async with aiofiles.open(filename, "a", encoding="utf-8") as f:
            line = json.dumps(result, ensure_ascii=False)
            await f.write(line + "\n")
            await f.flush()
    except Exception as e:
        logger.warning(f"⚠️ Failed to append to {filename.name}: {e}")


async def process_all_documents(
    llm,
    documents: List[str],
    model_name: str = "gemini-2.0-flash-lite",
    output_file: str = "summaries.jsonl",
    concurrency: int = 5,
):
    """
    Runs summarization concurrently (limited by `concurrency`),
    writes each result immediately (JSONL format),
    and displays progress with tqdm.
    """
    output_path = Path(output_file).resolve()
    failed_path = output_path.with_name("failed_summaries.jsonl")

    print(f"Starting parallel summarization of {len(documents)} documents...")
    print(f"Output file: {output_path}")

    semaphore = asyncio.Semaphore(concurrency)
    failed_documents: List[Dict[str, Any]] = []

    async def worker(idx: int, doc: str):
        async with semaphore:
            try:
                result = await summarize_document(llm=llm, document=doc, model_name=model_name)
                await append_jsonl(result, output_path)
            except Exception as e:
                failed_entry = {"document": doc, "error": str(e)}
                failed_documents.append(failed_entry)
                await append_jsonl(failed_entry, failed_path)
            finally:
                await asyncio.sleep(random.uniform(0.05, 0.1))

    tasks = [worker(i + 1, doc) for i, doc in enumerate(documents)]
    await tqdm_asyncio.gather(*tasks, total=len(tasks), desc="Summarizing", leave=True)

    print("\n--- Processing Complete ---")
    print(f"✅ Results written incrementally to {output_path}")
    if failed_documents:
        print(f"⚠️ {len(failed_documents)} failed → {failed_path}")
    else:
        print("✅ All documents processed successfully!")

    return {"failed": failed_documents, "output_file": str(output_path)}


In [10]:
selected_data = merged_data[: len(merged_data) // 2]

len(selected_data)

38782

In [25]:
# selected_data = merged_data[len(merged_data) // 2 + 1: ]

# save_json = {
#     "document": selected_data
# }

# save_data_path = "../data/vn_part_dataset_002.json"
# write_json(save_data_path, save_json)

Successfully wrote data to ../data/vn_part_dataset_002.json


True

In [ ]:
output_file = str(DATA_DIR / "vn_sum_dataset_001.jsonl")
try:
    results = await process_all_documents(llm=client, 
                                        documents=selected_data, 
                                        model_name="gemini-2.0-flash-lite", 
                                        output_file=output_file)
except KeyboardInterrupt:
    print("\nProcess interrupted by user.")
except Exception as e:
    print(f"An unexpected error occurred during execution: {e}")


"""
NOTE: 
    - 25/10/2025: vn_sum_dataset_001 - 18294/38782
        - something wrong with the async => too slow.
"""

Starting parallel summarization of 38782 documents...
Output file: /home/octoopt/workspace/projects/personal/data_enrichment/data/vn_sum_dataset_001.jsonl


Summarizing:  47%|████▋     | 18294/38782 [2:05:16<2:20:18,  2.43it/s] 


CancelledError: 

In [15]:
from datasets import load_dataset, DatasetDict
from huggingface_hub import HfApi, HfFolder
import pandas as pd
from pathlib import Path

# ---------------- CONFIG ----------------
JSONL_PATH = DATA_DIR / "vn_sum_dataset_001.jsonl" # your JSONL output file
REPO_ID = "8Opt/vietnamese-summarization-dataset-001"  # <- change this
SPLIT_RATIO = [0.8, 0.1, 0.1]  # train/val/test

# ---------------- LOAD DATA ----------------
# Hugging Face can directly load JSONL
dataset = load_dataset("json", data_files=str(JSONL_PATH))["train"]
print(f"✅ Loaded {len(dataset)} samples")

# ---------------- CLEAN DATA ----------------
# Optional cleaning — drop incomplete rows
keep_cols = ["document", "summary", "keywords"]
dataset = dataset.filter(lambda x: all(x.get(c) for c in keep_cols))
print(f"🧹 After cleaning: {len(dataset)} samples")

# ---------------- SPLIT DATA ----------------
# Shuffle before split for randomness
dataset = dataset.shuffle(seed=42)

# 80% train, 10% val, 10% test
train_testvalid = dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": train_testvalid["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

print(dataset_dict)
print({
    k: len(v)
    for k, v in dataset_dict.items()
})

# ---------------- PUSH TO HUB ----------------
# Login (if not already)
HfFolder.save_token(HF_TOKEN)
api = HfApi()

# Create dataset repo if not exist
api.create_repo(REPO_ID, repo_type="dataset", exist_ok=True)

# Push all splits at once
dataset_dict.push_to_hub(REPO_ID)
print(f"🚀 Successfully pushed dataset with 80/10/10 split to:")
print(f"🔗 https://huggingface.co/datasets/{REPO_ID}")


Generating train split: 18436 examples [00:00, 125377.08 examples/s]


✅ Loaded 18436 samples


Filter: 100%|██████████| 18436/18436 [00:00<00:00, 39603.80 examples/s]


🧹 After cleaning: 18436 samples
DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 14748
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1844
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1844
    })
})
{'train': 14748, 'validation': 1844, 'test': 1844}


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  1.86ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   2%|▏         |  524kB / 28.5MB,  291kB/s  




Processing Files (0 / 1)                :   4%|▎         | 1.05MB / 28.5MB,  375kB/s  







Processing Files (0 / 1)                :   6%|▌         | 1.57MB / 28.5MB,  358kB/s  




Processing Files (0 / 1)                :   7%|▋         | 2.10MB / 28.5MB,  388kB/s  


Processing Files (0 / 1)                :   9%|▉         | 2.62MB / 28.5MB,  437kB/s  

Processing Files (0 / 1)                :  11%|█         | 3.15MB / 28.5MB,  492kB/s  



Processing Files (0 / 1)                :  13%|█▎        | 3.67MB / 28.5MB,  510kB/s  


Processing Files (0 / 1)                :  15%|█▍        | 4.20MB / 28.5MB,  538kB/s  



Processing Files (0 / 1)                :  17%|█▋        | 4.72MB / 28.5MB,  549kB/s  

Processing Files (0 / 1)

🚀 Successfully pushed dataset with 80/10/10 split to:
🔗 https://huggingface.co/datasets/8Opt/vietnamese-summarization-dataset-001


## Build NER-POS

In [ ]:
sentence = "Bác sĩ bây giờ có thể thản nhiên báo tin bệnh nhân bị ung thư"

word_tokenize(sentence)

In [2]:
word_tokenize(sentence, format="text")

'Bác_sĩ bây_giờ có_thể thản_nhiên báo tin bệnh_nhân bị ung_thư'

In [6]:
import os
from datasets import load_dataset
from dotenv import load_dotenv


load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

In [7]:
from openai import OpenAI

# 1. Initialize the OpenAI client, but configure it for Gemini
client = OpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY, 
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/" 
)

# 2. Make a chat completion request as you normally would, 
# but specify a compatible Gemini model (e.g., "gemini-2.5-flash")
response = client.chat.completions.create(
    model="gemini-2.5-flash", 
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain to me how AI works"}
    ]
)

print(response.choices[0].message.content)

At its core, **AI (Artificial Intelligence)** is about enabling machines to **simulate human intelligence**. This doesn't mean they think exactly like us, but rather they can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, understanding language, and recognizing patterns.

The most common and impactful way AI works today, especially in what's called **Machine Learning (a subset of AI)**, is through **learning from data**. Instead of being explicitly programmed for every single scenario, AI models learn to make predictions or decisions based on patterns they identify in vast amounts of information.

Let's break it down into simpler steps:

### The Core Idea: Learning from Experience

Imagine you want to teach a child to identify cats.
1.  **You show them many pictures:** Some pictures are of cats, some are of dogs, birds, or other animals.
2.  **You label them:** "This is a cat," "This is a dog," "This is a cat."
3.  **The chi

In [9]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY, 
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/" 
)

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.beta.chat.completions.parse(
    model="gemini-2.0-flash",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "John and Susan are going to an AI conference on Friday."},
    ],
    response_format=CalendarEvent,
)

print(completion.choices[0].message.parsed)

name='AI conference' date='Friday' participants=['John', 'Susan']
